El propósito de este notebook es evaluar la calidad de los datos, realizar las transformaciones necesarias y generar datasets limpios para el análisis exploratorio, la base de datos SQL y el modelo de Machine Learning.

In [1]:
import pandas as pd
import numpy as np

In [2]:
customers_df = pd.read_csv ('olist_customers_dataset.csv')
geolocation_df = pd.read_csv ('olist_geolocation_dataset.csv')
order_items_df = pd.read_csv ('olist_order_items_dataset.csv')
order_payments_df = pd.read_csv ('olist_order_payments_dataset.csv')
order_reviews_df = pd.read_csv ('olist_order_reviews_dataset.csv')
orders_df = pd.read_csv ('olist_orders_dataset.csv')
products_df = pd.read_csv ('olist_products_dataset.csv')
sellers_df = pd.read_csv ('olist_sellers_dataset.csv')
category_translation_df = pd.read_csv ('product_category_name_translation.csv')

In [3]:
#Genero una copia para no trabajar sobre el dataframe original
customers = customers_df.copy()
geolocation = geolocation_df.copy()
order_items = order_items_df.copy()
order_payments = order_payments_df.copy()
order_reviews = order_reviews_df.copy()
orders = orders_df.copy()
products = products_df.copy()
sellers = sellers_df.copy()
category_translation = category_translation_df.copy()

# IDENTIFICACION DE COLUMNAS FECHA Y TRANSFORMACION

In [4]:
order_payments.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [5]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])


In [6]:
orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [7]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

In [8]:
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])

In [9]:
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])

In [10]:
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

In [11]:
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])

In [12]:
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

In [13]:
orders.head() #Verificamos la conversion

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


# Verificacion de PKs

In [14]:
customers["customer_id"].duplicated().sum() #No hay duplicados en pk de esta tabla

np.int64(0)

In [15]:
order_items["order_id"].duplicated().sum() #Es esperable este resultado, ya que varios productos pueden integrar la misma orden.

np.int64(13984)

In [16]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum() #Esta seria la verificacion correcta de las PK. Ya que la combinacion de la orden y el id del item no deberia repetirse

np.int64(0)

In [17]:
order_reviews['review_id'].duplicated().sum() 

np.int64(814)

In [18]:
order_reviews[
    order_reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


Tras analizar los registros, se observa que una misma reseña puede estar asociada a más de un pedido dentro del dataset, por lo que review_id no puede considerarse una clave primaria única. Se evaluará la combinación (review_id, order_id) como identificador del registro.

In [19]:
order_reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

np.int64(0)

In [20]:
orders['order_id'].duplicated().sum() # No se presentan duplicados en la PK de esta tabla

np.int64(0)

In [21]:
products['product_id'].duplicated().sum() # No se presentan duplicados en la PK

np.int64(0)

In [22]:
sellers['seller_id'].duplicated().sum() # No se presentan duplicados en la PK

np.int64(0)

# Analisis de Nulos

In [23]:
customers.isnull().sum()

customers.isnull().mean()*100

customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64

In [24]:
geolocation.isnull().sum()

geolocation.isnull().mean()*100

geolocation_zip_code_prefix    0.0
geolocation_lat                0.0
geolocation_lng                0.0
geolocation_city               0.0
geolocation_state              0.0
dtype: float64

In [25]:
order_items.isnull().sum()

order_items.isnull().mean()*100

order_id               0.0
order_item_id          0.0
product_id             0.0
seller_id              0.0
shipping_limit_date    0.0
price                  0.0
freight_value          0.0
dtype: float64

In [26]:
order_payments.isnull().sum()

order_payments.isnull().mean()*100

order_id                0.0
payment_sequential      0.0
payment_type            0.0
payment_installments    0.0
payment_value           0.0
dtype: float64

In [27]:
order_reviews.isnull().sum()

order_reviews.isnull().mean()*100 

review_id                   0.000000
order_id                    0.000000
review_score                0.000000
review_comment_title       88.341530
review_comment_message     58.702532
review_creation_date        0.000000
review_answer_timestamp     0.000000
dtype: float64

## Valores nulos contemplados ya que no todos los clientes escriben una reseña del producto.

In [28]:
orders.isnull().sum()

orders.isnull().mean()*100

order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
dtype: float64

## Decision recomendada, investigar estos pedidos que no fueron entregados al cliente y/o a la empresa de entregas. Representacion numerica "4.77%"

In [29]:
# Creamos categorías operativas según dónde se interrumpe la fecha
condiciones = [
    orders['order_delivered_customer_date'].notna(),
    orders['order_delivered_carrier_date'].notna() & orders['order_delivered_customer_date'].isna(),
    orders['order_approved_at'].notna() & orders['order_delivered_carrier_date'].isna(),
]
opciones = [
    '1. Completado (Entregado)', 
    '2. En Camino (Transportista)', 
    '3. En Preparación / Bodega'
]

orders['fase_pedido'] = pd.Series(
    pd.Series(
        np.select(condiciones, opciones, default='4. Cancelado / Pago No Aprobado')
    )
)

# Resumen ejecutivo para toma de decisiones
resumen = orders.groupby('fase_pedido').agg(
    total_pedidos=('order_id', 'count'),
    estados_comunes=('order_status', lambda x: x.unique().tolist())
)
resumen['porcentaje'] = (resumen['total_pedidos'] / len(orders)) * 100

print(resumen[['total_pedidos', 'porcentaje', 'estados_comunes']])

                                 total_pedidos  porcentaje  \
fase_pedido                                                  
1. Completado (Entregado)                96476   97.018332   
2. En Camino (Transportista)              1183    1.189650   
3. En Preparación / Bodega                1636    1.645197   
4. Cancelado / Pago No Aprobado            146    0.146821   

                                                                   estados_comunes  
fase_pedido                                                                         
1. Completado (Entregado)                                    [delivered, canceled]  
2. En Camino (Transportista)                        [shipped, canceled, delivered]  
3. En Preparación / Bodega       [invoiced, processing, unavailable, canceled, ...  
4. Cancelado / Pago No Aprobado                                [canceled, created]  


### Segun el analisis, se demostró que un 2.8% de los valores no entregados ni a cliente, ni a empresa de logistica se deben a las distintas fases del pedido, demostradas en la tabla.

## Diagnóstico para la Toma de Decisiones
1. La fuga principal está ANTES del envío (2.98%)
De ese 4.77% total, más del 60% (el 2.98%) ni siquiera llegó a entregarse a la empresa de transporte.

Causa: Pedidos cancelados a tiempo por el usuario, pagos no aprobados, o cuellos de botella en almacén/empaque.

Acción de Negocio: Enfocar esfuerzos en la pasarela de pagos y la operación del almacén. Aquí es donde el negocio pierde ventas directas.

2. La fricción logística externa es menor (1.79%)
Solo un 1.79% de los pedidos fue entregado a la mensajería pero no registra recepción del cliente.

Causa: Paquetes actualmente en ruta hacia el cliente, extravíos de la logística externa, o falta de actualización en el sistema por parte del repartidor.

Acción de Negocio: Auditar el SLA (acuerdo de nivel de servicio) de las empresas de transporte para identificar si hay paquetes atascados o extraviados.

In [30]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [31]:
productos_sin_peso = products['product_weight_g'].isna() | products['product_weight_g'] == 0

In [32]:
productos_sin_peso[productos_sin_peso]

9769     True
13683    True
14997    True
32079    True
Name: product_weight_g, dtype: bool

In [33]:
# Vemos los productos carentes de peso. Decision, pedir al vendedor la especificacion.
products[productos_sin_peso]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [34]:
productos_sin_longitud = products['product_length_cm'].isna() | products['product_length_cm'] == 0

In [35]:
productos_sin_longitud[productos_sin_longitud]

Series([], Name: product_length_cm, dtype: bool)

In [36]:
productos_sin_altura = products['product_height_cm'].isna() | products['product_height_cm'] == 0

In [37]:
productos_sin_altura[productos_sin_altura]

Series([], Name: product_height_cm, dtype: bool)

In [38]:
productos_sin_profundidad = products['product_width_cm'].isna() | products['product_width_cm'] == 0

In [39]:
productos_sin_profundidad[productos_sin_profundidad]

Series([], Name: product_width_cm, dtype: bool)

## No hay productos sin dimensiones, solo algunos pocos sin peso, acciones especificas con el vendedor.

In [40]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,fase_pedido
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1. Completado (Entregado)
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1. Completado (Entregado)
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1. Completado (Entregado)
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1. Completado (Entregado)
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1. Completado (Entregado)


In [41]:
fechas_imposibles = (orders['order_purchase_timestamp'] > orders['order_delivered_customer_date']).sum()

print(f"fechas_imposibles: {fechas_imposibles}")

fechas_imposibles: 0


#### No se presentan fechas imposibles

# Creacion de variables relevantes para el analisis y posterior algoritmo

# NOTA SOBRE IDENTIFICADORES DE CLIENTE:
# 'customer_id' es una clave temporal única por cada PEDIDO/TRANSACCIÓN.
# 'customer_unique_id' identifica a la PERSONA REAL (deduplicada por DNI/email).
# -> Para métricas de ventas y logística usar: customer_id
# -> Para métricas de retención, LTV y clientes recurrentes usar: customer_unique_id

In [42]:
# Dataset base para el modelo de Machine Learning
model_df = orders.copy()

# Agregar customer_unique_id
model_df = model_df.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

### Variables Entrega

In [43]:
# Tiempo de entrega
model_df["delivery_time_days"] = (
    model_df["order_delivered_customer_date"] -
    model_df["order_purchase_timestamp"]
).dt.days

In [44]:
#Dias de atraso
model_df["delivery_delay_days"] = (
    model_df["order_delivered_customer_date"] -
    model_df["order_estimated_delivery_date"]
).dt.days

In [45]:
# Define si la entrega esta o estuvo atrasada
model_df["is_delayed"] = (
    model_df["delivery_delay_days"] > 0
).astype(int)

### Variables Estacionalidad

In [46]:
# Mes de compra
model_df["purchase_month"] = (
    model_df["order_purchase_timestamp"]
    .dt.month
)

In [47]:
# Dia de la semana que se compró
model_df["purchase_week_day"] = (
    model_df["order_purchase_timestamp"]
    .dt.weekday
)

In [48]:
# Hora de compra
model_df["purchase_hour"] = (
    model_df["order_purchase_timestamp"]
    .dt.hour
)

In [49]:
# Trimestre de compra
model_df["purchase_quarter"] = (
    model_df["order_purchase_timestamp"]
    .dt.quarter
)

### Variables Precio/Costos

In [50]:
# Precio total orden
order_value = (
    order_items
    .groupby("order_id")["price"]
    .sum()
    .reset_index(name="total_order_value")
)

model_df = model_df.merge(
    order_value,
    on="order_id",
    how="left"
)

In [51]:
# Costo de envio
freight = (
    order_items
    .groupby("order_id")["freight_value"]
    .sum()
    .reset_index(name="total_freight")
)

model_df = model_df.merge(
    freight,
    on="order_id",
    how="left"
)

In [52]:
# Items por pedido
items = (
    order_items
    .groupby("order_id")
    .size()
    .reset_index(name="items_per_order")
)

model_df = model_df.merge(
    items,
    on="order_id",
    how="left"
)

In [53]:
# Porcentaje de valor del envio
model_df["freight_ratio"] = (
    model_df["total_freight"] /
    model_df["total_order_value"]
)

### Variables cliente

In [54]:
# Total gastado por el cliente
customer_spent = (
    model_df
    .groupby("customer_unique_id")["total_order_value"]
    .sum()
    .reset_index(name="customer_total_spent")
)

model_df = model_df.merge(
    customer_spent,
    on="customer_unique_id",
    how="left"
)

In [55]:
# Promedio gastado por cliente x pedido
customer_avg = (
    model_df
    .groupby("customer_unique_id")["total_order_value"]
    .mean()
    .reset_index(name="customer_avg_ticket")
)

model_df = model_df.merge(
    customer_avg,
    on="customer_unique_id",
    how="left"
)

In [56]:
customer_orders = (
    model_df
    .groupby("customer_unique_id")
    .agg(
        customer_total_orders=("order_id", "count")
    )
    .reset_index()
)

In [57]:
model_df = model_df.merge(
    customer_orders,
    on="customer_unique_id",
    how="left"
)

In [58]:
model_df["is_new_customer"] = (
    model_df["customer_total_orders"] == 1
).astype(int)

In [59]:
# Tiempo que paso desde la primera compra del cliente
first_purchase = (
    model_df
    .groupby("customer_unique_id")
    .agg(
        first_purchase=("order_purchase_timestamp", "min")
    )
    .reset_index()
)

model_df = model_df.merge(
    first_purchase,
    on="customer_unique_id",
    how="left"
)

model_df["customer_days_since_first_purchase"] = (
    model_df["order_purchase_timestamp"] -
    model_df["first_purchase"]
).dt.days

model_df.drop(columns="first_purchase", inplace=True)

In [60]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [61]:
#Se unifican reviews debido al descubrimiento de dos reseñas por el mismo cliente hacia el mismo producto, decidimos quedarnos con la ultima segun fecha de registro.
reviews_unique = (
    order_reviews
    .sort_values("review_answer_timestamp")
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
)

In [62]:
model_df = model_df.merge(
    reviews_unique[
        ["order_id","review_score"]
    ],
    on="order_id",
    how="left"
)

### Variables Vendedor

In [63]:
main_seller = (
    order_items
    .groupby("order_id")
    .first()
    .reset_index()[["order_id", "seller_id"]]
)

model_df = model_df.merge(
    main_seller,
    on="order_id",
    how="left"
)

In [64]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


In [65]:
# Reviews del vendedor
seller_reviews = (
    order_items
    .merge(
        order_reviews[["order_id","review_score"]],
        on="order_id",
        how="left"
    )
)
# Review promedio del vendedor
seller_avg_review = (
    seller_reviews
    .groupby("seller_id")["review_score"]
    .mean()
    .reset_index(name="seller_avg_review")
)

In [66]:
# Envios del vendedor
seller_delivery = (
    order_items
    .merge(
        model_df[
            ["order_id", "delivery_time_days"]
        ],
        on="order_id",
        how="left"
    )
)
# Envio promedio del vendedor
seller_avg_delivery = (
    seller_delivery
    .groupby("seller_id")["delivery_time_days"]
    .mean()
    .reset_index(name="seller_avg_delivery")
)

In [67]:
model_df = model_df.merge(
    seller_avg_review,
    on="seller_id",
    how="left"
)

model_df = model_df.merge(
    seller_avg_delivery,
    on="seller_id",
    how="left"
)

In [68]:
# Cuotas
installments = (
    order_payments
    .groupby("order_id")["payment_installments"]
    .max()
    .reset_index()
)

model_df = model_df.merge(
    installments,
    on="order_id",
    how="left"
)

In [69]:
# Tamaño/Volumen final del producto
products["product_volume"] = (
    products["product_length_cm"] *
    products["product_height_cm"] *
    products["product_width_cm"]
)

In [70]:
product_volume = (
    order_items
    .merge(
        products[
            ["product_id", "product_volume"]
        ],
        on="product_id",
        how="left"
    )
    .groupby("order_id")["product_volume"]
    .mean()
    .reset_index()
)

model_df = model_df.merge(
    product_volume,
    on="order_id",
    how="left"
)

In [71]:
# Ordenes totales por vendedor
seller_total_orders = (
    order_items
    .groupby("seller_id")
    .agg(
        seller_total_orders=("order_id", "nunique")
    )
    .reset_index()
)

model_df = model_df.merge(
    seller_total_orders,
    on="seller_id",
    how="left"
)

In [72]:
# Ventas totales por vendedor
seller_total_sales = (
    order_items
    .groupby("seller_id")
    .agg(
        seller_total_sales=("price", "sum")
    )
    .reset_index()
)

model_df = model_df.merge(
    seller_total_sales,
    on="seller_id",
    how="left"
)

In [73]:
model_df = model_df.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

In [74]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99441 entries, 52798 to 70017
Data columns (total 34 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   order_id                            99441 non-null  object        
 1   customer_id                         99441 non-null  object        
 2   order_status                        99441 non-null  object        
 3   order_purchase_timestamp            99441 non-null  datetime64[ns]
 4   order_approved_at                   99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date        97658 non-null  datetime64[ns]
 6   order_delivered_customer_date       96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date       99441 non-null  datetime64[ns]
 8   fase_pedido                         99441 non-null  object        
 9   customer_unique_id                  99441 non-null  object        
 10  delivery_time_days     

## Variable Objetivo:

In [75]:
model_df["target"] = (
    model_df["review_score"] >= 4
).astype(int)

In [76]:
model_df[["target"]].sample()

,target
37724,1


In [77]:
model_df.to_csv('Data_ML_EDA.csv', index=False)